# POD analysis with modal-correlation reduction support

This notebook extends the original force POD workflow by adding modal coefficient correlation analysis so redundant modes can be identified before building a surrogate model.

In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Union
from dataclasses import dataclass, fields

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebook plotting
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True


In [ ]:
@dataclass
class PODResult:
    file: str
    rpm: int
    torque: float
    n_teeth: int
    n_angles: int
    singular_values: np.ndarray
    cum_energy_fraction: np.ndarray
    energy_loss_vs_modes: np.ndarray
    residual_energy_ratio_vs_modes: np.ndarray
    peak_residual_vs_modes_temporal: np.ndarray
    U: np.ndarray
    Vt: np.ndarray
    A: np.ndarray
    angle_index: pd.Index
    tooth_index: np.ndarray
    fs_hz: Optional[float]
    temporal_fmax_hz: Optional[float]
    export: Optional[Dict[str, Path]]
    modal_coefficients: Optional[np.ndarray] = None
    mode_correlation_matrix: Optional[np.ndarray] = None
    correlated_mode_pairs: Optional[List[Tuple[int, int, float]]] = None
    effective_independent_modes: Optional[int] = None

    def __getitem__(self, key: str):
        if not hasattr(self, key):
            raise KeyError(key)
        return getattr(self, key)

    def __iter__(self):
        for f in fields(self):
            yield f.name

    def __len__(self):
        return len(fields(self))

    def to_dict(self) -> Dict[str, object]:
        return {f.name: getattr(self, f.name) for f in fields(self)}


In [ ]:
class PODRadialForces:
    def __init__(
        self,
        sheet_name: Union[str, int] = "Torque_Characteristic",
        out_dir: Union[str, Path] = "pod_outputs",
        max_modes: Optional[int] = None,
    ) -> None:
        self.sheet_name = sheet_name
        self.out_dir = Path(out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self.max_modes = max_modes

    @staticmethod
    def parse_operating_conditions(file_path: str) -> Tuple[int, float]:
        name = Path(file_path).stem.lower()
        m = re.search(r"(\d+)\s*rpm[^\d]+(\d+(?:\.\d+)?)\s*nm", name, re.IGNORECASE)
        if not m:
            compact = re.sub(r"[^a-z0-9]+", "", name)
            m = re.search(r"(\d+)rpm(\d+(?:\.\d+)?)nm", compact)
        return (int(m.group(1)), float(m.group(2))) if m else (-1, float("nan"))

    @staticmethod
    def _normalize_col(c: str) -> str:
        c2 = str(c).strip().replace(" ", "_")
        c2 = c2.replace("Force_Tangential_", "FT_")
        c2 = c2.replace("Force_Radial_", "FR_")
        if c2.lower().startswith("rotor_position") or c2 == "Elec.Deg.":
            return "Rotor_Angle_elec"
        return c2

    def _read_force_dataframe_with_time(self, xlsx_path: str) -> Tuple[pd.DataFrame, Optional[np.ndarray]]:
        df = pd.read_excel(xlsx_path, sheet_name=self.sheet_name, engine="openpyxl")
        df = df.dropna(axis=1, how="all")
        df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
        if df.shape[0] > 0:
            if df.iloc[0].astype(str).str.contains(r"Seconds|Elec\.Deg\.|Mech\.Deg\.|Nm|N", regex=True).any():
                df = df.iloc[1:].copy()
        df.columns = [self._normalize_col(c) for c in df.columns]
        df = df.loc[:, ~df.columns.duplicated()]

        time_col = None
        for cand in df.columns:
            if str(cand).lower().startswith("time"):
                time_col = cand
                break

        df = df.apply(pd.to_numeric, errors="coerce")

        if "Rotor_Angle_elec" in df.columns:
            df = df.set_index("Rotor_Angle_elec")
        else:
            df.index.name = "sample"

        time_s = None
        if time_col is not None and time_col in df.columns:
            time_s = df[time_col].to_numpy(dtype=float, copy=True)

        fr_cols = [c for c in df.columns if str(c).startswith("FR_")]
        if not fr_cols:
            raise ValueError(f"No FR_* columns found in {xlsx_path}")

        def tooth_id(c: str) -> int:
            m = re.search(r"FR_(\d+)", c)
            return int(m.group(1)) if m else 10**9

        fr_cols = sorted(fr_cols, key=tooth_id)
        df_fr = df[fr_cols].copy()
        df_fr = df_fr.dropna(axis=0, how="any")
        return df_fr, time_s

    @staticmethod
    def _svd(A: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        return np.linalg.svd(A, full_matrices=False)

    @staticmethod
    def _reconstruct(U: np.ndarray, s: np.ndarray, Vt: np.ndarray, k: int) -> np.ndarray:
        Uk, sk, Vtk = U[:, :k], s[:k], Vt[:k, :]
        return (Uk * sk) @ Vtk

    @staticmethod
    def _energy_curves(s: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        e = s**2
        frac = np.cumsum(e) / e.sum()
        loss = 1.0 - frac
        return frac, loss

    @staticmethod
    def _fft_peak_non_dc_1d_band_by_bin(x: np.ndarray, idx_min: int = 1, idx_max: Optional[int] = None) -> float:
        sig = np.asarray(x).ravel()
        spec = np.fft.rfft(sig)
        mag = np.abs(spec)
        n_bins = mag.size
        lo = max(1, int(idx_min))
        hi = int(idx_max) if idx_max is not None else n_bins - 1
        hi = max(lo, min(hi, n_bins - 1))
        band = mag[lo:hi + 1]
        return float(band.max()) if band.size else 0.0

    @staticmethod
    def _temporal_bins_from_fmax(n_samples: int, fmax_hz: Optional[float], fs_hz: Optional[float]) -> Tuple[int, Optional[int]]:
        if fmax_hz is None or fs_hz is None or fs_hz <= 0:
            return 1, None
        n_bins = (n_samples // 2) + 1
        df = fs_hz / n_samples
        k_max = int(np.floor(fmax_hz / df))
        k_max = max(1, min(k_max, n_bins - 1))
        return 1, k_max

    def _fft_residual_metrics_temporal(
        self,
        A: np.ndarray,
        R: np.ndarray,
        temporal_fmax_hz: Optional[float],
        fs_hz: Optional[float],
    ) -> Dict[str, float]:
        denom = np.linalg.norm(A, "fro") ** 2
        numer = np.linalg.norm(R, "fro") ** 2
        ratio = float(numer / denom) if denom > 0 else np.nan
        n_angles = R.shape[1]
        lo, hi = self._temporal_bins_from_fmax(n_angles, temporal_fmax_hz, fs_hz)
        peaks = [self._fft_peak_non_dc_1d_band_by_bin(row, lo, hi) for row in R]
        return {
            "residual_energy_ratio": ratio,
            "peak_residual_temporal": float(np.mean(peaks)) if peaks else np.nan,
        }

    def _fft_residual_temporal_curve(
        self,
        A: np.ndarray,
        U: np.ndarray,
        s: np.ndarray,
        Vt: np.ndarray,
        k_vals: np.ndarray,
        temporal_fmax_hz: Optional[float],
        fs_hz: Optional[float],
    ) -> np.ndarray:
        out: List[float] = []
        n_angles = A.shape[1]
        lo, hi = self._temporal_bins_from_fmax(n_angles, temporal_fmax_hz, fs_hz)
        for k in k_vals:
            R = A - self._reconstruct(U, s, Vt, int(k))
            row_peaks = [self._fft_peak_non_dc_1d_band_by_bin(row, lo, hi) for row in R]
            out.append(float(np.mean(row_peaks)))
        return np.asarray(out, dtype=float)

    def _fft_residual_spatial_curve(
        self,
        A: np.ndarray,
        U: np.ndarray,
        s: np.ndarray,
        Vt: np.ndarray,
        k_vals: np.ndarray,
    ) -> np.ndarray:
        out: List[float] = []
        n_teeth = A.shape[0]
        lo, hi = 1, (n_teeth // 2)
        for k in k_vals:
            R = A - self._reconstruct(U, s, Vt, int(k))
            col_peaks = [self._fft_peak_non_dc_1d_band_by_bin(R[:, j], lo, hi) for j in range(R.shape[1])]
            out.append(float(np.mean(col_peaks)))
        return np.asarray(out, dtype=float)

    @staticmethod
    def _modal_coefficients(s: np.ndarray, Vt: np.ndarray, n_modes: Optional[int] = None) -> np.ndarray:
        r = len(s) if n_modes is None else min(int(n_modes), len(s))
        return np.diag(s[:r]) @ Vt[:r, :]

    @staticmethod
    def compute_mode_correlation(modal_coefficients: np.ndarray) -> np.ndarray:
        coeff = np.asarray(modal_coefficients, dtype=float)
        if coeff.ndim != 2:
            raise ValueError("modal_coefficients must be a 2D array of shape (modes, samples)")
        if coeff.shape[0] == 1:
            return np.array([[1.0]], dtype=float)
        corr = np.corrcoef(coeff)
        corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
        np.fill_diagonal(corr, 1.0)
        return corr

    @staticmethod
    def find_correlated_mode_pairs(corr_matrix: np.ndarray, threshold: float = 0.90) -> List[Tuple[int, int, float]]:
        pairs: List[Tuple[int, int, float]] = []
        n = corr_matrix.shape[0]
        for i in range(n):
            for j in range(i + 1, n):
                val = float(corr_matrix[i, j])
                if abs(val) >= threshold:
                    pairs.append((i + 1, j + 1, val))
        return pairs

    @staticmethod
    def effective_mode_rank(modal_coefficients: np.ndarray, tol: Optional[float] = None) -> int:
        return int(np.linalg.matrix_rank(modal_coefficients, tol=tol))

    def mode_correlation_dataframe(
        self,
        s: np.ndarray,
        Vt: np.ndarray,
        n_modes: Optional[int] = None,
    ) -> pd.DataFrame:
        coeff = self._modal_coefficients(s, Vt, n_modes=n_modes)
        corr = self.compute_mode_correlation(coeff)
        labels = [f"Mode_{i+1}" for i in range(corr.shape[0])]
        return pd.DataFrame(corr, index=labels, columns=labels)

    def plot_mode_correlation(
        self,
        corr_matrix: np.ndarray,
        title: str = "Correlation Matrix Between POD Modes",
        figsize: Tuple[int, int] = (9, 7),
        annotate: bool = True,
    ) -> None:
        corr_matrix = np.asarray(corr_matrix, dtype=float)
        fig, ax = plt.subplots(figsize=figsize)
        im = ax.imshow(corr_matrix, vmin=-1.0, vmax=1.0, cmap="coolwarm", aspect="auto")
        n = corr_matrix.shape[0]
        labels = [f"M{i+1}" for i in range(n)]
        ax.set_xticks(np.arange(n))
        ax.set_yticks(np.arange(n))
        ax.set_xticklabels(labels, rotation=45, ha="right")
        ax.set_yticklabels(labels)
        ax.set_title(title)
        ax.set_xlabel("Mode index")
        ax.set_ylabel("Mode index")
        if annotate:
            for i in range(n):
                for j in range(n):
                    ax.text(j, i, f"{corr_matrix[i, j]:.2f}", ha="center", va="center", fontsize=8)
        cbar = fig.colorbar(im, ax=ax)
        cbar.set_label("Correlation coefficient")
        plt.tight_layout()
        plt.show()

    def analyze_file(
        self,
        xlsx_path: str,
        make_plots: bool = False,
        save_csv: bool = True,
        k_export: Optional[int] = None,
        temporal_fmax_hz: Optional[float] = None,
        fs_hz_override: Optional[float] = None,
        correlation_threshold: float = 0.90,
        corr_modes: Optional[int] = None,
    ) -> PODResult:
        rpm, torque = self.parse_operating_conditions(xlsx_path)
        df_fr, time_s = self._read_force_dataframe_with_time(xlsx_path)
        A = df_fr.to_numpy(dtype=float).T

        fs_hz: Optional[float] = None
        if fs_hz_override is not None and fs_hz_override > 0:
            fs_hz = float(fs_hz_override)
        elif time_s is not None and len(time_s) >= 2:
            dt = np.diff(time_s)
            if np.all(np.isfinite(dt)) and np.median(dt) > 0:
                fs_hz = 1.0 / float(np.median(dt))

        U, s, Vt = self._svd(A)
        cum, loss = self._energy_curves(s)
        max_modes = self.max_modes or min(A.shape)
        k_vals = np.arange(1, max_modes + 1)

        energy_loss_k: List[float] = []
        resid_ratio_k: List[float] = []
        peak_temporal_k: List[float] = []
        for k in k_vals:
            R = A - self._reconstruct(U, s, Vt, int(k))
            energy_loss_k.append(loss[int(k) - 1])
            met = self._fft_residual_metrics_temporal(A, R, temporal_fmax_hz=temporal_fmax_hz, fs_hz=fs_hz)
            resid_ratio_k.append(met["residual_energy_ratio"])
            peak_temporal_k.append(met["peak_residual_temporal"])

        corr_modes = max_modes if corr_modes is None else min(int(corr_modes), len(s))
        modal_coeff = self._modal_coefficients(s, Vt, n_modes=corr_modes)
        corr_matrix = self.compute_mode_correlation(modal_coeff)
        correlated_pairs = self.find_correlated_mode_pairs(corr_matrix, threshold=correlation_threshold)
        effective_rank = self.effective_mode_rank(modal_coeff)

        export_info: Optional[Dict[str, Path]] = None
        if save_csv:
            export_info = self._export_modes(
                xlsx_path=xlsx_path,
                rpm=rpm,
                torque=torque,
                df=df_fr,
                U=U,
                s=s,
                Vt=Vt,
                k_export=k_export,
                corr_matrix=corr_matrix,
                correlated_pairs=correlated_pairs,
            )

        angle_index = pd.Index(df_fr.index.values, name="Elec.Deg.")
        tooth_index = np.arange(1, U.shape[0] + 1)
        result = PODResult(
            file=Path(xlsx_path).name,
            rpm=rpm,
            torque=torque,
            n_teeth=A.shape[0],
            n_angles=A.shape[1],
            singular_values=s,
            cum_energy_fraction=cum,
            energy_loss_vs_modes=np.asarray(energy_loss_k),
            residual_energy_ratio_vs_modes=np.asarray(resid_ratio_k),
            peak_residual_vs_modes_temporal=np.asarray(peak_temporal_k),
            U=U,
            Vt=Vt,
            A=A,
            angle_index=angle_index,
            tooth_index=tooth_index,
            fs_hz=fs_hz,
            temporal_fmax_hz=temporal_fmax_hz,
            export=export_info,
            modal_coefficients=modal_coeff,
            mode_correlation_matrix=corr_matrix,
            correlated_mode_pairs=correlated_pairs,
            effective_independent_modes=effective_rank,
        )

        if make_plots and plt is not None:
            self._plot_summary(result)
            self.plot_mode_correlation(corr_matrix)
        return result

    def run_batch(
        self,
        data_dir: str = ".",
        pattern: str = "Torque_Characteristics_*rpm_*Nm*.xls*",
        make_plots: bool = False,
        save_csv: bool = True,
        k_export: Optional[int] = None,
        temporal_fmax_hz: Optional[float] = None,
        fs_hz_override: Optional[float] = None,
        correlation_threshold: float = 0.90,
        corr_modes: Optional[int] = None,
    ) -> List[PODResult]:
        files = sorted([str(p) for p in Path(data_dir).glob(pattern)])
        results: List[PODResult] = []
        for f in files:
            try:
                out = self.analyze_file(
                    f,
                    make_plots=make_plots,
                    save_csv=save_csv,
                    k_export=k_export,
                    temporal_fmax_hz=temporal_fmax_hz,
                    fs_hz_override=fs_hz_override,
                    correlation_threshold=correlation_threshold,
                    corr_modes=corr_modes,
                )
                print(
                    f"Processed {out.file} | RPM={out.rpm} | Torque={out.torque} Nm | "
                    f"teeth={out.n_teeth} | angles={out.n_angles} | fs={out.fs_hz} | "
                    f"effective independent modes={out.effective_independent_modes}"
                )
                results.append(out)
            except Exception as e:
                print(f"[WARN] {Path(f).name}: {e}")
        return results

    def _export_modes(
        self,
        xlsx_path: str,
        rpm: int,
        torque: float,
        df: pd.DataFrame,
        U: np.ndarray,
        s: np.ndarray,
        Vt: np.ndarray,
        k_export: Optional[int] = None,
        corr_matrix: Optional[np.ndarray] = None,
        correlated_pairs: Optional[List[Tuple[int, int, float]]] = None,
    ) -> Dict[str, Path]:
        base = Path(xlsx_path).stem
        op_dir = self.out_dir / f"{rpm}rpm_{torque}Nm"
        op_dir.mkdir(parents=True, exist_ok=True)

        n_teeth, _ = U.shape
        V = Vt.T
        r = len(s)
        k = r if (k_export is None or k_export > r) else int(k_export)

        teeth_idx = pd.Index(np.arange(1, n_teeth + 1), name="Tooth")
        angle_idx = pd.Index(df.index.values, name="Elec.Deg.")

        if U[:, :k].shape[0] != len(teeth_idx):
            raise ValueError(f"U rows {U[:, :k].shape[0]} != len(teeth_idx) {len(teeth_idx)}")
        if V[:, :k].shape[0] != len(angle_idx):
            print(f"[WARN] angle index length {len(angle_idx)} != V rows {V[:, :k].shape[0]}; falling back to RangeIndex.")
            angle_idx = pd.Index(np.arange(V[:, :k].shape[0]), name="Elec.Deg.")

        U_df = pd.DataFrame(U[:, :k], index=teeth_idx, columns=[f"Mode_{i+1}" for i in range(k)])
        S_df = pd.DataFrame({"Mode": np.arange(1, k + 1), "Sigma": s[:k]})
        V_df = pd.DataFrame(V[:, :k], index=angle_idx, columns=[f"Mode_{i+1}" for i in range(k)])

        U_path = op_dir / f"{base}_U_modes_{k}.csv"
        S_path = op_dir / f"{base}_S_sigma_{k}.csv"
        V_path = op_dir / f"{base}_V_modes_{k}.csv"
        U_df.to_csv(U_path, index=True)
        S_df.to_csv(S_path, index=False)
        V_df.to_csv(V_path, index=True)

        exported = {"U": U_path, "S": S_path, "V": V_path}

        if corr_matrix is not None:
            corr_labels = [f"Mode_{i+1}" for i in range(corr_matrix.shape[0])]
            corr_df = pd.DataFrame(corr_matrix, index=corr_labels, columns=corr_labels)
            corr_path = op_dir / f"{base}_mode_correlation_matrix.csv"
            corr_df.to_csv(corr_path, index=True)
            exported["corr_matrix"] = corr_path

        if correlated_pairs is not None:
            pair_df = pd.DataFrame(correlated_pairs, columns=["Mode_i", "Mode_j", "Correlation"])
            pairs_path = op_dir / f"{base}_highly_correlated_mode_pairs.csv"
            pair_df.to_csv(pairs_path, index=False)
            exported["corr_pairs"] = pairs_path

        return exported

    def correlation_summary(self, res: PODResult) -> pd.DataFrame:
        if res.correlated_mode_pairs is None:
            return pd.DataFrame(columns=["Mode_i", "Mode_j", "Correlation"])
        return pd.DataFrame(res.correlated_mode_pairs, columns=["Mode_i", "Mode_j", "Correlation"])

    def _plot_summary(self, res: PODResult) -> None:
        fig, axes = plt.subplots(2, 2, figsize=(12, 9))
        idx = np.arange(1, len(res.singular_values) + 1)

        axes[0, 0].plot(idx, res.singular_values, "o-")
        axes[0, 0].set_title("Singular values")
        axes[0, 0].set_xlabel("Mode index")
        axes[0, 0].set_ylabel("σᵢ")

        axes[0, 1].plot(idx, res.cum_energy_fraction, "o-")
        axes[0, 1].set_title("Cumulative energy fraction")
        axes[0, 1].set_xlabel("# modes")
        axes[0, 1].set_ylabel("Energy captured")

        k_vals = np.arange(1, len(res.energy_loss_vs_modes) + 1)
        axes[1, 0].plot(k_vals, res.energy_loss_vs_modes, "o-")
        axes[1, 0].set_title("Energy loss vs # modes")
        axes[1, 0].set_xlabel("# modes")
        axes[1, 0].set_ylabel("1 - cumulative energy")

        axes[1, 1].plot(k_vals, res.peak_residual_vs_modes_temporal, "o-", label="Temporal FFT residual")
        axes[1, 1].plot(k_vals, res.residual_energy_ratio_vs_modes, "o-", label="Residual energy ratio")
        axes[1, 1].set_title("Residual metrics")
        axes[1, 1].set_xlabel("# modes")
        axes[1, 1].set_ylabel("Residual value")
        axes[1, 1].legend()

        fig.suptitle(f"POD summary | {res.file} | {res.rpm} rpm | {res.torque} Nm", fontsize=12)
        plt.tight_layout()
        plt.show()


In [ ]:
# Example usage
# Update data_dir and file_name for your machine.

data_dir = Path('.')
file_name = 'Torque_Characteristics_1000rpm_75Nm.xlsx'

pod = PODRadialForces(sheet_name='Torque_Characteristic', out_dir='pod_outputs')

res = pod.analyze_file(
    xlsx_path=str(data_dir / file_name),
    make_plots=True,
    save_csv=True,
    temporal_fmax_hz=3900.0,
    fs_hz_override=3900.0,
    correlation_threshold=0.90,
    corr_modes=10,   # correlation check only on first 10 retained modal coefficient signals
)

print('Used fs [Hz]:', res.fs_hz)
print('Effective independent modes from correlation block:', res.effective_independent_modes)
print('Exported files:', res.export)

corr_df = pod.mode_correlation_dataframe(res.singular_values, res.Vt, n_modes=10)
display(corr_df)

corr_pairs_df = pod.correlation_summary(res)
display(corr_pairs_df)
